# 13 — M&A Target Screener
> Score every acquisition candidate against your own criteria — strategic fit, financial fit, and operational fit — and get a ranked shortlist with investment thesis, risks, and next steps in seconds.

*Run all cells. Swap the input in the final code cell with your own data.*

In [ ]:
%pip install -q langchain-openai langchain-core pydantic python-dotenv
import os
os.environ['OPENAI_API_KEY'] = 'sk-...'  # replace

In [ ]:
from typing import List, Literal, Optional
from pydantic import BaseModel, Field
from langchain_core.messages import HumanMessage, SystemMessage
from langchain_openai import ChatOpenAI


# --- Data structures ---

class DimensionScore(BaseModel):
    score: int = Field(description="0-10 score for this dimension")
    rationale: str = Field(description="Evidence-based reason for the score")
    meets_threshold: bool = Field(description="True if this dimension clears the minimum bar")


class TargetAssessmentCard(BaseModel):
    company_name: str
    sector: str
    geography: str
    strategic_fit: DimensionScore = Field(
        description="How well the target fits the acquirer's strategic direction"
    )
    financial_fit: DimensionScore = Field(
        description="Revenue size, EBITDA margin, growth profile vs. rubric"
    )
    operational_fit: DimensionScore = Field(
        description="Integration complexity, management quality, cultural alignment"
    )
    overall_score: int = Field(description="Composite 0-30")
    recommendation: Literal["proceed", "monitor", "pass"]
    investment_thesis: str = Field(
        description="One sentence on why this target creates value, or why it doesn't"
    )
    key_risks: List[str] = Field(description="Top 2-3 deal risks")
    suggested_next_step: str


class ScreeningResult(BaseModel):
    """Ranked M&A shortlist from multi-criteria screening."""
    acquirer: Optional[str] = None
    rubric_summary: str = Field(
        description="One sentence summarising the screening criteria applied"
    )
    shortlist: List[TargetAssessmentCard] = Field(
        description="Targets that scored above threshold, ranked by overall_score descending"
    )
    screened_out: List[str] = Field(
        description="Company names that failed one or more threshold criteria"
    )
    recommendation: str = Field(
        description="Top-line view on the most attractive target and why"
    )


# --- Scoring logic ---

SCREENER_SYSTEM = SystemMessage(
    """You are a senior M&A analyst at a private equity firm.

You are given a description of an acquirer's strategic criteria and a list of
acquisition targets. For EACH target, score it across three dimensions (0-10 each):

  - strategic_fit   : sector alignment, geography, business model match
  - financial_fit   : revenue size, EBITDA margin, growth vs. the stated rubric
  - operational_fit : integration complexity, management depth, cultural alignment

Threshold: a score below 5 on ANY dimension is a fail -- that target is screened out.

overall_score = strategic_fit + financial_fit + operational_fit  (max 30)

recommendation per target:
  - "proceed"  if overall_score >= 20 and no dimension below 5
  - "monitor"  if overall_score 14-19 and no dimension below 5
  - "pass"     if any dimension is below 5 OR overall_score < 14

Rank the shortlist by overall_score descending.
Set meets_threshold = True only when the dimension score >= 5.

Be precise and evidence-based. Quote the specific numbers from the brief."""
)


def run(screening_brief: str) -> ScreeningResult:
    llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)
    screener = SCREENER_SYSTEM | llm.with_structured_output(ScreeningResult)
    return screener.invoke(
        HumanMessage(content="Screening brief and targets:\n\n" + screening_brief)
    )

## The scenario

A US growth equity firm is searching for a healthcare technology bolt-on for one of its portfolio companies. They have set clear acquisition criteria — revenue band, margin floor, geography, and integration timeline — and shortlisted five candidates ranging from a high-margin compliance tool to a loss-making telehealth platform.

The agent scores each candidate on three dimensions, filters out anyone who misses a threshold, and returns a ranked shortlist with a one-sentence investment thesis and a concrete next step for every candidate that clears the bar.

In [ ]:
BRIEF = """
ACQUIRER: Meridian Growth Equity

Meridian is a US growth equity fund seeking a bolt-on acquisition for its
healthcare technology platform company. Screening criteria:

  - Sector       : Healthcare SaaS or digital health infrastructure (no direct patient care)
  - Revenue      : USD 8m - 40m ARR or subscription revenue
  - EBITDA margin: 12%+ (demonstrable path to profitability acceptable)
  - Geography    : USA or Canada only
  - Integration  : Must plug into existing EHR/API layer within 9 months

TARGETS FOR SCREENING:

1. CareCompliance Inc
   Geography: USA  |  Sector: Healthcare HR compliance SaaS
   Revenue: USD 14m  |  EBITDA margin: 18%  |  Growth: +27% YoY
   Notes: 88% recurring revenue, 115 NRR, SOC 2 Type II certified, 55 employees

2. NorthStar Health Analytics
   Geography: Canada  |  Sector: Population health analytics SaaS
   Revenue: USD 9m  |  EBITDA margin: 14%  |  Growth: +19% YoY
   Notes: Provincial government contracts (85% of revenue), limited API documentation

3. TeleMed Direct Corp
   Geography: USA  |  Sector: Telehealth platform (direct patient care)
   Revenue: USD 32m  |  EBITDA margin: -7%  |  Growth: +41% YoY
   Notes: High growth but loss-making; heavily regulated, requires clinical staff licensing

4. RxFlow Systems
   Geography: USA  |  Sector: Pharmacy workflow automation SaaS
   Revenue: USD 21m  |  EBITDA margin: 20%  |  Growth: +22% YoY
   Notes: 94% gross retention, deep EHR integrations already live, 70 employees, ISO 27001

5. BioTrack Canada Ltd
   Geography: Canada  |  Sector: Clinical trial data management SaaS
   Revenue: USD 6m  |  EBITDA margin: 11%  |  Growth: +15% YoY
   Notes: Below revenue floor; margin just under threshold; niche but strategic product
"""

result = run(BRIEF)

print("=" * 65)
print(f"M&A SCREENING RESULT | Acquirer: {result.acquirer or 'Meridian Growth Equity'}")
print("=" * 65)
print(f"\nRubric: {result.rubric_summary}")
print(f"\nTop-line: {result.recommendation}")

print(f"\nSHORTLIST ({len(result.shortlist)} targets):")
for t in result.shortlist:
    print(f"\n  [{t.overall_score}/30] {t.company_name} ({t.geography})")
    print(f"  Recommendation: {t.recommendation.upper()}")
    print(f"  Thesis: {t.investment_thesis}")
    print(
        f"  Scores -- Strategic: {t.strategic_fit.score}/10 "
        f"| Financial: {t.financial_fit.score}/10 "
        f"| Operational: {t.operational_fit.score}/10"
    )
    print(f"  Key risks: {', '.join(t.key_risks)}")
    print(f"  Next step: {t.suggested_next_step}")

if result.screened_out:
    print(f"\nSCREENED OUT: {', '.join(result.screened_out)}")

## Use your own data

Replace the `BRIEF` string above with:
- Your acquirer profile: sector focus, revenue range, margin floor, geography, and integration timeline
- Your candidate list: company name, geography, sector, revenue, EBITDA margin, growth rate, and any notable operating details

The agent will return a ranked shortlist with a score on each dimension, a one-sentence investment thesis, the top risks, and a suggested next step for every candidate that clears your thresholds.